In [ ]:

from src.datadreamer import DataDreamer
from src.llms import OpenAI
from src.steps import DataFromPrompt, ProcessWithPrompt
from src.trainers import TrainHFFineTune
from src.steps.data_sources.csv_data_source import CSVDataSource
import openai

import os
from dotenv import load_dotenv
load_dotenv()
INDOX_API_KEY= os.getenv("INDOX_API_KEY")


with DataDreamer("./output"):
   # Load GPT-4
   gpt_4 = OpenAI(api_key=INDOX_API_KEY,model_name="gpt-4o-mini")

   # Generate synthetic arXiv-style research paper abstracts with GPT-4
   arxiv_dataset = DataFromPrompt(
      "Generate Research Paper Abstracts",
      args={
         "llm": gpt_4,
         "n": 7,
         "temperature": 1.2,
         "instruction": (
            "Generate an arXiv abstract of an NLP research paper."
            " Return just the abstract, no titles."
         ),
      },
      outputs={"generations": "abstracts"},
   )

   # Ask GPT-4 to convert the abstracts to tweets
   abstracts_and_tweets = ProcessWithPrompt(
      "Generate Tweets from Abstracts",
      inputs={"inputs": arxiv_dataset.output["abstracts"]},
      args={
         "llm": gpt_4,
         "instruction": (
            "Given the abstract, write a tweet to summarize the work."
         ),
         "top_p": 1.0,
      },
      outputs={"inputs": "abstracts", "generations": "tweets"},
   )

   # Create training data splits
   splits = abstracts_and_tweets.splits(train_size=0.90, validation_size=0.10)

   # Train a model to convert research paper abstracts to tweets
   # with the synthetic dataset
   trainer = TrainHFFineTune(
      "Train an Abstract => Tweet Model",
      model_name="google/t5-v1_1-base",
   )
   trainer.train(
      train_input=splits["train"].output["abstracts"],
      train_output=splits["train"].output["tweets"],
      validation_input=splits["validation"].output["abstracts"],
      validation_output=splits["validation"].output["tweets"],
      epochs=30,
      batch_size=8,
   )

   # Publish and share the synthetic dataset
   abstracts_and_tweets.publish_to_hf_hub(
      "datadreamer-dev/abstracts_and_tweets",
      train_size=0.90,
      validation_size=0.10,
   )

   # Publish and share the trained model
   trainer.publish_to_hf_hub("datadreamer-dev/abstracts_to_tweet_model")

[ 🤖 DataDreamer 💤 ] Initialized. 🚀 Dreaming to folder: ./output
[ 🤖 DataDreamer 💤 ] Step 'Generate Research Paper Abstracts' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'Generate Research Paper Abstracts' finished and is saved to disk. 🎉
[ 🤖 DataDreamer 💤 ] Step 'Generate Tweets from Abstracts' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'Generate Tweets from Abstracts' finished and is saved to disk. 🎉
[ 🤖 DataDreamer 💤 ] Step 'Generate Tweets from Abstracts (train split)' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'Generate Tweets from Abstracts (train split)' finished and is saved to disk. 🎉
[ 🤖 DataDreamer 💤 ] Step 'Generate Tweets from Abstracts (validation split)' is running. ⏳
[ 🤖 DataDreamer 💤 ] Step 'Generate Tweets from Abstracts (validation split)' finished and is saved to disk. 🎉
C:\Users\RayanSabz\Downloads\indoxJudge-feature-visualization_2\venv\lib\site-packages\huggingface_hub\file_download.py:159: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently sto

'Train an Abstract => Tweet Model / Tokenize Train Input':   0%|          | 0/6 [00:00<?, ? examples/s]

C:\Users\RayanSabz\Downloads\indoxJudge-feature-visualization_2\venv\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
[ 🤖 DataDreamer 💤 ] Step 'Train an Abstract => Tweet Model / Tokenize Train Input' finished and is saved to disk. 🎉
[ 🤖 DataDreamer 💤 ] Step 'Train an Abstract => Tweet Model / Tokenize Train Output' is running. ⏳


'Train an Abstract => Tweet Model / Tokenize Train Output':   0%|          | 0/6 [00:00<?, ? examples/s]

[ 🤖 DataDreamer 💤 ] Step 'Train an Abstract => Tweet Model / Tokenize Train Output' finished and is saved to disk. 🎉
[ 🤖 DataDreamer 💤 ] Step 'Train an Abstract => Tweet Model / Tokenize Validation Input' is running. ⏳


'Train an Abstract => Tweet Model / Tokenize Validation Input':   0%|          | 0/1 [00:00<?, ? examples/s]

[ 🤖 DataDreamer 💤 ] Step 'Train an Abstract => Tweet Model / Tokenize Validation Input' finished and is saved to disk. 🎉
[ 🤖 DataDreamer 💤 ] Step 'Train an Abstract => Tweet Model / Tokenize Validation Output' is running. ⏳


'Train an Abstract => Tweet Model / Tokenize Validation Output':   0%|          | 0/1 [00:00<?, ? examples/s]

[ 🤖 DataDreamer 💤 ] Step 'Train an Abstract => Tweet Model / Tokenize Validation Output' finished and is saved to disk. 🎉
